# 00.7 — pip, packaging, and the anatomy of a Python project

You've installed this project with `pip install -e ".[dev]"` — this notebook explains every character of that command, and everything around it: where packages come from, how versions are pinned, what `pyproject.toml` declares, why the code lives under `src/`, and how `python -m neural_data_decoding` finds its way to `cli.py`.

By the end you can read any modern Python project's structure at a glance — and (in [00.8](00.8_build_a_dl_project_from_scratch.ipynb)) build your own.

**Prerequisites:** [00.5 the command line](00.5_the_command_line_for_matlab_users.ipynb); [01.5 modules and imports](../01_python_for_matlab_users/01.5_modules_and_imports.ipynb) helps but isn't required.

## Section 1 — What MATLAB does

MATLAB dependencies are **toolboxes**: installed through the Add-On manager into one global MATLAB installation, versioned with MATLAB itself (R2023b's Deep Learning Toolbox, full stop). Your own reusable code is folders on the MATLAB path.

Consequences you've likely lived with:

* Two projects needing different toolbox versions? Install two MATLABs.
* Sharing code = sharing folders + a README saying which toolboxes to install.
* No manifest records what a project depends on; you discover missing toolboxes at runtime.

Python inverts all three: dependencies are **per-project** (the venv), declared **in a manifest** (`pyproject.toml`), and installed from a **public index** (PyPI) with explicit versions.

## Section 2 — The concepts you need

### 2.1 — pip and PyPI

**PyPI** (the Python Package Index, [pypi.org](https://pypi.org)) hosts ~500k packages. **pip** is the installer that fetches from it — always into whichever Python environment pip itself belongs to (this is why activation matters; [00.5 §2.5](00.5_the_command_line_for_matlab_users.ipynb)).

The commands you'll actually use:

| Command | Does |
|---|---|
| `pip install torch` | install newest compatible version + its dependencies |
| `pip install "torch==2.2.2"` | install an exact version |
| `pip install --upgrade torch` | upgrade to newest |
| `pip uninstall torch` | remove |
| `pip list` | everything installed in this environment |
| `pip show torch` | version, location, dependencies of one package |
| `pip freeze` | machine-readable list of everything, with exact versions |

In [ ]:
import sys
!{sys.executable} -m pip show torch | head -6

(Note the `python -m pip` form — it guarantees pip and Python are the same environment. Typing bare `pip` occasionally grabs a different environment's pip; the `-m` form can't.)

### 2.2 — Version specifiers

Declaring a dependency means declaring *which versions are acceptable*:

| Specifier | Means |
|---|---|
| `torch` | any version (avoid — today's code, tomorrow's breakage) |
| `torch==2.2.2` | exactly 2.2.2 (reproducible, but never gets fixes) |
| `torch>=2.2` | 2.2 or newer (the common library choice) |
| `torch~=2.2.0` | >=2.2.0 but <2.3 ("compatible release") |

The convention behind these numbers is **semantic versioning** — `MAJOR.MINOR.PATCH`, where a MAJOR bump signals breaking changes, MINOR adds features, PATCH fixes bugs. Not every package honors it perfectly, but it's the shared vocabulary.

### 2.3 — `pyproject.toml`: the project manifest

One file at the project root declares everything about a modern Python project. The real one from this repo:

In [ ]:
!sed -n '1,40p' ../../pyproject.toml

Reading it top to bottom:

* **`[build-system]`** — what tool builds the package (boilerplate; setuptools is the default choice).
* **`[project]`** — name, version, description, `requires-python` floor.
* **`dependencies`** — what `pip install` pulls in automatically. This is the answer to MATLAB's undeclared-toolbox problem: the manifest IS the requirements list.
* **`[project.optional-dependencies]`** — named extras. `dev` here bundles the development tools (pytest, jupyter, matplotlib, nbstripout…). Installing `".[dev]"` = "this project, plus the dev extra". End users who just want the library skip the extras.
* Further down (open the file!): **`[tool.pyright]`**, **`[tool.pytest.ini_options]`**, **`[tool.interrogate]`** — per-tool configuration blocks. One file, one place to look.

### 2.4 — `requirements.txt` vs `pyproject.toml`

You'll meet `requirements.txt` in older tutorials — a plain list of `package==version` lines installed with `pip install -r requirements.txt`. The modern split:

* **`pyproject.toml`** declares *ranges* — what the project is compatible with.
* **`pip freeze > requirements-lock.txt`** records *exact* versions of everything currently installed — a **lockfile** snapshot for exactly reproducing an environment (e.g., "the paper's results used these versions").

Both have a place: manifest for development, freeze-file for archival reproducibility.

### 2.5 — The `src/` layout and editable installs

This project's tree — the standard modern shape:

```
neural_data_decoding/          ← repo root
├── pyproject.toml             ← manifest (above)
├── src/
│   └── neural_data_decoding/  ← THE package (what you import)
│       ├── __init__.py
│       ├── cli.py
│       ├── data/ ...
├── tests/                     ← tests OUTSIDE the package
├── configs/                   ← YAML configs
├── notebooks/                 ← this curriculum
└── .venv/                     ← the environment (gitignored)
```

Why the extra `src/` level? It forces every import to go through the *installed* package rather than accidentally through the working directory — which means your tests exercise the same import path your users get. (One-line reason to copy the convention; the long version is in the further reading.)

**Editable installs** connect the two worlds. `pip install -e .` doesn't copy the code into `site-packages` — it plants a *link* pointing back at `src/`. Edit a `.py` file, and the change is live at the next import (or kernel restart) with no reinstall. That's the mode you always want during development, and it's why every edit you make to this repo's source is immediately visible in these notebooks:

In [ ]:
import neural_data_decoding
# The package resolves back into src/ in the repo — not a site-packages copy:
print(neural_data_decoding.__file__)

### 2.6 — Entry points: how `python -m neural_data_decoding` works

`python -m <package>` runs the package's `__main__.py`. Ours glues the module system to the CLI — note the `if __name__ == "__main__":` guard, which ensures the CLI only fires when the file is *run*, not when a documentation tool merely *imports* it (the docstring explains exactly this):

In [ ]:
!cat ../../src/neural_data_decoding/__main__.py

So the training commands you've seen (`python -m neural_data_decoding train ...`) are: interpreter → `__main__.py` → `cli.main()` → argparse ([01.8 §8](../01_python_for_matlab_users/01.8_the_python_standard_library_for_matlab_users.ipynb)). No magic left.

## Section 3 — The `neural_data_decoding` implementation

The one-screen summary of how the pieces you've now met interlock in this repo:

| Piece | Role |
|---|---|
| `pyproject.toml [project].dependencies` | torch, numpy, scipy, mat73, omegaconf… — what training needs |
| `[project.optional-dependencies].dev` | pytest, jupyter, matplotlib, nbstripout, interrogate — what development needs |
| `pip install -e ".[dev]"` | link the package + install both lists into `.venv` |
| `src/` layout | imports always resolve through the installed (linked) package |
| `__main__.py` | makes the package runnable as `python -m neural_data_decoding` |
| `[tool.*] blocks` | configure pyright / pytest / interrogate without extra config files |
| `.venv/` in `.gitignore` | environments are rebuilt from the manifest, never committed |

That last row is the reproducibility contract: **commit the manifest, never the environment.** Anyone (including the ACCRE cluster) rebuilds byte-equivalent-enough environments with two commands: create venv, `pip install -e ".[dev]"`.

## Section 4 — Hands-on exercises

### Exercise 4.1 — interrogate the environment

Using pip (in this kernel via `!{sys.executable} -m pip ...`): find which version of `mat73` is installed and what IT depends on.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
import sys
!{sys.executable} -m pip show mat73

### Exercise 4.2 — build and editable-install a package from scratch

The complete minimal package — manifest + one module — created, installed, imported. This is the skeleton 00.8 grows into a real project:

In [ ]:
import subprocess, sys, pathlib

pkg = pathlib.Path("/tmp/ndd_minipkg")
(pkg / "src" / "minipkg").mkdir(parents=True, exist_ok=True)

(pkg / "pyproject.toml").write_text('''
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "minipkg"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = []

[tool.setuptools.packages.find]
where = ["src"]
''')
(pkg / "src" / "minipkg" / "__init__.py").write_text('def hello():\n    return "minipkg lives!"\n')

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(pkg)],
                   capture_output=True, text=True)
print(r.stderr or "installed OK")

# One wrinkle: editable installs register themselves via a .pth file that
# Python processes at INTERPRETER STARTUP. A package installed mid-session
# normally needs a kernel restart to become importable. For this demo we
# poke the site machinery to re-scan instead:
import site, importlib
site.main()
importlib.invalidate_caches()

import minipkg
print(minipkg.hello())
print("resolves to:", minipkg.__file__)

In [ ]:
# Cleanup so this demo doesn't linger in your venv:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "minipkg"])
print("uninstalled")

## Section 5 — Common errors

### `pip: command not found` — or packages install but imports fail

pip belongs to a different environment than the Python running your code. Diagnose with `which pip` vs `which python`; cure permanently by always using `python -m pip ...`.

### `error: externally-managed-environment`

You ran pip against the OS's system Python (Homebrew/Debian protect theirs). You never want that anyway — create and activate a venv first.

### `ERROR: ResolutionImpossible` / dependency conflicts

Two requirements can't be satisfied together. Read pip's report of which packages clash; usually one pin needs loosening. A fresh venv per project prevents most of these — cross-project conflicts can't exist when environments aren't shared.

### Edits to source have no effect

Three suspects, in order: (1) the package was installed *without* `-e`, so you're running a stale copy — `pip show <pkg>` reveals the location; (2) a notebook kernel is holding the old import — restart it; (3) two copies are installed and the wrong one wins — `pip uninstall` until `pip show` reports the path you expect.

### `ModuleNotFoundError` after a successful install

Installed into one environment, importing from another — THE classic. `print(sys.executable)` in the failing context, compare against `which pip`. (Full kernel-side flowchart: [00.4 §5](00.4_ide_deep_dive.ipynb).)

## Section 6 — Further reading

- [Python Packaging User Guide](https://packaging.python.org/en/latest/tutorials/packaging-projects/) — the official end-to-end tutorial.
- [pyproject.toml specification](https://packaging.python.org/en/latest/specifications/pyproject-toml/) — every legal key.
- [src layout vs flat layout](https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/) — the long version of §2.5's one-liner.
- [semver.org](https://semver.org/) — semantic versioning spelled out.

Next up: **[00.8 build a DL project from scratch](00.8_build_a_dl_project_from_scratch.ipynb)** — everything from 00.5–00.7 combined into creating a working deep-learning project from an empty directory.